# 01b — Batch Ingestion
**Ingest all past year papers from subject folders — with image extraction**

```
data/papers/
├── instrumentation_and_control/
│   ├── sept_2025.pdf
│   ├── jan_2025.pdf
│   └── sept_2024.pdf
└── another_subject/
    └── sept_2025.pdf
```
Per paper: Gemini extraction → parent-child chunks → Ollama embed
→ BM25 sparse → Pinecone upsert → PostgreSQL parents → Supabase images

---
## Section 1 — Setup

In [11]:
import os, json, pickle, base64, time, hashlib, re
from pathlib import Path
from collections import defaultdict, deque
from dotenv import load_dotenv
load_dotenv()

import fitz                                      
import google.generativeai as genai
import ollama
import psycopg2
from psycopg2.extras import Json
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from supabase import create_client
from dataclasses import dataclass, field
from typing import Optional

# ── Clients ──────────────────────────────────────────────────────────────────
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model    = genai.GenerativeModel('gemma-4-31b-it')

test = model.generate_content(
    "Reply with just: ok",
    generation_config={"response_mime_type": "application/json"},
)
raw = (test.text or "").strip()
if not raw:
    raise ValueError("Gemini returned empty response text")
print("Gemini:", raw)

pc       = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index    = pc.Index('papersloth')
supabase = create_client(os.getenv('SUPABASE_URL'), os.getenv('SUPABASE_KEY'))

# ── Constants ─────────────────────────────────────────────────────────────────
EMBED_MODEL          = 'nomic-embed-text-v2-moe:latest'
BUCKET               = 'exam-images'
PAPERS_ROOT          = Path('data/papers')
CHECKPOINT           = Path('data/checkpoint.json')
BM25_PATH            = Path('data/bm25_model.pkl')
ALL_EMBED_TEXTS_PATH = Path('data/all_embed_texts.json')
Path('data').mkdir(exist_ok=True)

stats = index.describe_index_stats()
print(f'✅ Setup complete')
print(f'   Pinecone : {stats["total_vector_count"]} vectors')
print(f'   Supabase : bucket = {BUCKET}')

Gemini: *   User input: "Reply with just: ok"
    *   System constraint: "Only generate a valid, parseable JSON."
    *   System constraint: "Do not include any extra text outside of the JSON string."

    *   The user wants the word "ok".
    *   The system requires a JSON output.
    *   To satisfy both, I should wrap the response in a JSON object or array.

    *   Option 1: `{"response": "ok"}`
    *   Option 2: `["ok"]`

    *   Valid, parseable JSON.
    *   Double-quoted strings.
    *   No comments.
    *   Single JSON block.
    *   No extra text.

    *   `{"reply": "ok"}` or simply `["ok"]`. Since the user asked for "ok", putting it in a string within a JSON object is the safest bet.

    *   `{"result": "ok"}
`
{"reply": "ok"}
✅ Setup complete
   Pinecone : 0 vectors
   Supabase : bucket = exam-images


---
## Section 2 — Folder Scanner

In [12]:
def scan_papers(root: Path) -> list[dict]:
    papers = []
    for pdf in sorted(root.rglob('*.pdf')):
        subject = pdf.parent.name if pdf.parent != root else 'unknown'
        papers.append({
            'path':    str(pdf),
            'subject': subject,
            'name':    pdf.name,
            'key':     hashlib.md5(str(pdf).encode()).hexdigest()[:12],
        })
    return papers

if CHECKPOINT.exists():
    with open(CHECKPOINT) as f:
        done_keys = set(json.load(f).get('done', []))
else:
    done_keys = set()

all_papers = scan_papers(PAPERS_ROOT)
pending    = [p for p in all_papers if p['key'] not in done_keys]
already    = [p for p in all_papers if p['key'] in done_keys]

print(f'Found {len(all_papers)} PDFs  |  Already done: {len(already)}  |  Pending: {len(pending)}')
print()

by_subject = defaultdict(list)
for p in pending:
    by_subject[p['subject']].append(p['name'])
for subject, files in sorted(by_subject.items()):
    print(f'  📁 {subject}/')
    for f in files:
        print(f'       {f}')

eta = len(pending) * 5 / 60
print(f'\nEstimated time: ~{eta:.0f} min')

Found 11 PDFs  |  Already done: 0  |  Pending: 11

  📁 unknown/
       inc jan 2024.pdf
       inc jan2023.pdf
       inc jan2025.pdf
       inc may2022.pdf
       inc may2023.pdf
       inc may2024.pdf
       inc may2025.pdf
       inc sept2022.pdf
       inc sept2023.pdf
       inc sept2024.pdf
       inc sept2025.pdf

Estimated time: ~1 min


---
## Section 3 — Rate Limiter + Extraction Prompt

In [14]:
class RateLimiter:
    """Sliding-window rate limiter. Blocks until a call slot is free."""
    def __init__(self, max_calls: int = 14, window_secs: float = 60.0):
        self.max_calls   = max_calls   # 14 not 15 — leave one buffer slot
        self.window_secs = window_secs
        self._calls      = deque()

    def wait(self):
        now = time.time()
        while self._calls and now - self._calls[0] > self.window_secs:
            self._calls.popleft()
        if len(self._calls) >= self.max_calls:
            sleep_for = self.window_secs - (now - self._calls[0]) + 0.5
            if sleep_for > 0:
                print(f'    ⏳ rate limit — sleeping {sleep_for:.1f}s')
                time.sleep(sleep_for)
        self._calls.append(time.time())

    @property
    def usage(self):
        now = time.time()
        return sum(1 for t in self._calls if now - t <= self.window_secs)

limiter = RateLimiter(max_calls=14, window_secs=60)
print('✅ Rate limiter ready')

✅ Rate limiter ready


In [15]:
# Combined prompt — metadata + questions in ONE API call per paper
COMBINED_PROMPT = """This is a UTP (Universiti Teknologi PETRONAS) final exam paper.

Return a single JSON object with two keys: \"metadata\" and \"questions\".

\"metadata\": {
  \"course_code\":     \"RBB3013\",
  \"course_name\":     \"Instrumentation and Control\",
  \"semester\":        \"September 2025\",
  \"year\":            2025,
  \"duration_hours\":  3.0,
  \"total_questions\": 4
}

\"questions\": [
  {
    \"question_number\": \"1\",
    \"sub_question\":    \"a\",
    \"page_number\":     2,
    \"question_text\":   \"Full text...\",
    \"marks\":           10,
    \"has_figure\":      true,
    \"figure_label\":    \"FIGURE Q1b\",
    \"question_type\":   \"calculation\"
  }
]

Rules:
- Skip cover page, instructions, appendix
- Split sub-questions (a/b/c or i/ii/iii) into SEPARATE items
- Include full question text with all values and conditions
- Data tables → markdown format inside question_text
- question_type: calculation | theory | diagram | table
- has_figure = true if question references any FIGURE or TABLE
- page_number is 1-indexed
- marks / sub_question are null if not applicable

Return ONLY the JSON. No markdown fences. No extra text."""

print('✅ Extraction prompt ready')

✅ Extraction prompt ready


---
## Section 4 — Processing Functions

All helpers used in the batch loop.

In [16]:
# ── Data classes ──────────────────────────────────────────────────────────────
@dataclass
class ChildChunk:
    child_id:        str
    parent_id:       str
    question_number: str
    sub_question:    Optional[str]
    question_text:   str
    embed_text:      str
    marks:           Optional[int]
    has_figure:      bool
    figure_label:    Optional[str]
    question_type:   str
    course_code:     str
    semester:        str
    year:            int

@dataclass
class ParentChunk:
    parent_id:       str
    question_number: str
    full_text:       str
    total_marks:     Optional[int]
    children:        list
    course_code:     str
    semester:        str
    year:            int

print('✅ Data classes defined')

✅ Data classes defined


In [22]:
def extract_pdf(pdf_path: str) -> dict:
    """Call Gemini once - returns metadata + questions."""
    with open(pdf_path, 'rb') as f:
        pdf_b64 = base64.standard_b64encode(f.read()).decode()

    def _parse_json(text: str) -> dict:
        raw = (text or '').strip()
        if not raw:
            raise ValueError('Gemini returned empty response text')
        if raw.startswith('```'):
            raw = raw.split('\n', 1)[1].rsplit('```', 1)[0].strip()
        start = raw.find('{')
        if start == -1:
            raise ValueError('No JSON object start found')
        raw = raw[start:]
        idx = 1
        while idx < len(raw) and raw[idx].isspace():
            idx += 1
        if idx < len(raw) and raw[idx] not in ['"', '}']:
            raise ValueError('JSON object does not start with a quoted key')
        try:
            obj, _ = json.JSONDecoder().raw_decode(raw)
        except json.JSONDecodeError as exc:
            preview = raw[:400].replace('\n', ' ')
            raise ValueError(f'Gemini JSON parse failed: {exc}. Raw preview: {preview}') from exc
        if not isinstance(obj, dict):
            raise ValueError('Gemini did not return a JSON object')
        return obj

    def _call(prompt: str) -> str:
        limiter.wait()
        resp = model.generate_content(
            [
                {'mime_type': 'application/pdf', 'data': pdf_b64},
                prompt,
            ],
            generation_config={'response_mime_type': 'application/json'},
        )
        return resp.text

    strict_prompt = (
        COMBINED_PROMPT
        + "\n\nReturn ONLY the JSON object. Use double quotes for all keys and strings."
        + " No prose, no math, no preface."
    )
    last_text = _call(COMBINED_PROMPT)
    try:
        return _parse_json(last_text)
    except ValueError as exc:
        print(f'   ⚠️  Gemini returned non-JSON, retrying... ({exc})')
    last_text = _call(strict_prompt)
    try:
        return _parse_json(last_text)
    except ValueError as exc:
        print(f'   ⚠️  Gemini still non-JSON, attempting repair... ({exc})')

    repair_prompt = (
        COMBINED_PROMPT
        + "\n\nFix the following output into a single valid JSON object with keys \"metadata\" and \"questions\"."
        + " Use double quotes for all keys and strings. If a value is unknown, use null."
        + " Return ONLY the JSON object.\n\nOUTPUT TO FIX:\n"
        + (last_text or "")
    )
    repaired_text = _call(repair_prompt)
    return _parse_json(repaired_text)


def build_chunks(extracted: dict) -> tuple:
    """Build parent + child objects from Gemini output."""
    meta        = extracted['metadata']
    questions   = extracted['questions']
    course_code = meta['course_code']
    semester    = meta['semester']
    year        = meta['year']
    sem_slug    = semester.replace(' ', '')

    groups  = defaultdict(list)
    for q in questions:
        groups[q['question_number']].append(q)

    parents:  list[ParentChunk] = []
    children: list[ChildChunk]  = []

    for qnum, subs in groups.items():
        parent_id  = f'{course_code}_{year}_{sem_slug}__Q{qnum}'
        full_parts = []
        child_ids  = []
        total_m    = 0

        for sub in subs:
            sub_l    = sub.get('sub_question') or ''
            child_id = f'{parent_id}_{sub_l}' if sub_l else f'{parent_id}_main'
            full_parts.append(
                f"Q{qnum}{sub_l}: {sub['question_text']}" +
                (f" [{sub['marks']} marks]" if sub.get('marks') else '')
            )
            child_ids.append(child_id)
            total_m += sub.get('marks') or 0

            preamble   = (
                f"Course: {course_code} | Semester: {semester} | "
                f"Question {qnum}{sub_l} | "
                f"Type: {sub.get('question_type','unknown')} | "
                f"Marks: {sub.get('marks','unknown')}\n"
            )
            children.append(ChildChunk(
                child_id        = child_id,
                parent_id       = parent_id,
                question_number = qnum,
                sub_question    = sub.get('sub_question'),
                question_text   = sub['question_text'],
                embed_text      = preamble + sub['question_text'],
                marks           = sub.get('marks'),
                has_figure      = sub.get('has_figure', False),
                figure_label    = sub.get('figure_label'),
                question_type   = sub.get('question_type', 'theory'),
                course_code     = course_code,
                semester        = semester,
                year            = year,
            ))

        parents.append(ParentChunk(
            parent_id       = parent_id,
            question_number = qnum,
            full_text       = '\n\n'.join(full_parts),
            total_marks     = total_m or None,
            children        = child_ids,
            course_code     = course_code,
            semester        = semester,
            year            = year,
        ))

    return parents, children


def extract_and_upload_images(
    pdf_path: str,
    questions: list[dict],
    course_code: str,
    semester: str,
    year: int,
) -> dict:
    """
    For every question with has_figure=True:
      - Render that PDF page as a 2x-resolution PNG (PyMuPDF)
      - Upload to Supabase Storage
      - Return {figure_label: public_url}
    """
    sem_slug    = semester.replace(' ', '')
    pdf_doc     = fitz.open(pdf_path)
    figure_urls = {}

    for q in questions:
        if not q.get('has_figure'):
            continue
        page_no = q.get('page_number')
        label   = q.get('figure_label') or f"Q{q['question_number']}{q.get('sub_question','')}"

        if not page_no:
            continue

        safe_label   = re.sub(r'[^\w\-]', '_', label)
        storage_path = f'{course_code}/{year}/{sem_slug}/{safe_label}.png'

        # Render page as PNG (2x zoom = ~150 DPI, clear for diagrams)
        page      = pdf_doc[page_no - 1]
        pix       = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
        img_bytes = pix.tobytes('png')

        try:
            supabase.storage.from_(BUCKET).upload(
                path=storage_path,
                file=img_bytes,
                file_options={'content-type': 'image/png', 'upsert': 'true'},
            )
        except Exception:
            pass  # upsert=true handles duplicates; other errors caught by caller

        url = supabase.storage.from_(BUCKET).get_public_url(storage_path)
        figure_urls[label] = url

    pdf_doc.close()
    return figure_urls  # { 'FIGURE Q1b': 'https://...', ... }


def upsert_to_pinecone(
    children:    list[ChildChunk],
    sparse_vecs: list,
 ) -> None:
    """Embed each child (Ollama) and upsert dense+sparse to Pinecone."""
    vectors = []
    for child, sparse in zip(children, sparse_vecs):
        resp  = ollama.embed(model=EMBED_MODEL, input=f'search_document: {child.embed_text}')
        dense = resp['embeddings'][0]
        vectors.append({
            'id':            child.child_id,
            'values':        dense,
            'sparse_values': sparse,
            'metadata': {
                'parent_id':       child.parent_id,
                'question_number': child.question_number,
                'sub_question':    child.sub_question or '',
                'marks':           child.marks or 0,
                'has_figure':      child.has_figure,
                'question_type':   child.question_type,
                'course_code':     child.course_code,
                'semester':        child.semester,
                'year':            child.year,
                'text_preview':    child.question_text[:200],
            },
        })
    for i in range(0, len(vectors), 100):
        index.upsert(vectors=vectors[i:i+100])


def upsert_parents_to_postgres(
    parents:     list[ParentChunk],
    figure_urls: dict,
) -> None:
    """
    Store each parent in PostgreSQL.
    image_urls are matched to each parent by question number:
      'FIGURE Q1b' -> parent with question_number='1'
    """
    conn = psycopg2.connect(os.getenv('DATABASE_URL'))
    cur  = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS parent_chunks (
            parent_id TEXT PRIMARY KEY, question_number TEXT,
            full_text TEXT, total_marks INT, children JSONB,
            image_urls JSONB, course_code TEXT, semester TEXT,
            year INT, created_at TIMESTAMP DEFAULT NOW()
        );
    """)
    cur.execute("ALTER TABLE parent_chunks ADD COLUMN IF NOT EXISTS image_urls JSONB;")

    for p in parents:
        # Match figure labels that contain this question's number
        # e.g. 'FIGURE Q1b' matches question_number='1'
        q_imgs = {
            label: url
            for label, url in figure_urls.items()
            if re.search(rf'Q{p.question_number}[a-z]?', label, re.IGNORECASE)
        }
        cur.execute("""
            INSERT INTO parent_chunks
                (parent_id,question_number,full_text,total_marks,
                 children,image_urls,course_code,semester,year)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (parent_id) DO UPDATE SET
                full_text=EXCLUDED.full_text,
                total_marks=EXCLUDED.total_marks,
                children=EXCLUDED.children,
                image_urls=EXCLUDED.image_urls;
        """, (
            p.parent_id, p.question_number, p.full_text,
            p.total_marks, Json(p.children), Json(q_imgs),
            p.course_code, p.semester, p.year,
        ))

    conn.commit(); cur.close(); conn.close()


def save_checkpoint(done_keys: set) -> None:
    with open(CHECKPOINT, 'w') as f:
        json.dump({'done': list(done_keys)}, f)


print('✅ All functions defined')

✅ All functions defined


---
## Section 5 — Batch Ingestion Loop

Per paper:
1. Gemini extraction (1 API call — rate limited)
2. Build parent/child chunks
3. Upload figure images → Supabase
4. Embed children → Pinecone
5. Store parents → PostgreSQL
6. Save checkpoint

In [20]:
# Load existing embed texts for BM25 continuity
if ALL_EMBED_TEXTS_PATH.exists():
    with open(ALL_EMBED_TEXTS_PATH) as f:
        all_embed_texts: list[str] = json.load(f)
else:
    all_embed_texts = []

print(f'Starting. Existing corpus: {len(all_embed_texts)} texts.')

failed:  list[dict] = []
success: list[str]  = []

for paper in pending:
    path = paper['path']
    key  = paper['key']

    print(f"\n{'─'*60}")
    print(f"📄 {paper['subject']}/{paper['name']}  [RPM: {limiter.usage}/14]")

    try:
        t0 = time.time()

        # 1. Gemini — 1 call per paper ────────────────────────────────────
        extracted   = extract_pdf(path)
        meta        = extracted['metadata']
        course_code = meta['course_code']
        semester    = meta['semester']
        year        = meta['year']
        print(f'   ✅ Gemini  {course_code} {semester} '
              f'({len(extracted["questions"])} sub-q)  [{time.time()-t0:.1f}s]')

        # 2. Build chunks ──────────────────────────────────────────────────
        parents, children = build_chunks(extracted)
        new_texts = [c.embed_text for c in children]
        print(f'   ✅ Chunks  {len(parents)} parents  {len(children)} children')

        # 3. Images → Supabase ────────────────────────────────────────────
        figure_urls = extract_and_upload_images(
            path, extracted['questions'], course_code, semester, year
        )
        n_imgs = len(figure_urls)
        print(f'   ✅ Images  {n_imgs} uploaded to Supabase')

        # 4. BM25 sparse encoding ─────────────────────────────────────────
        all_embed_texts.extend(new_texts)
        bm25_local  = BM25Encoder()
        bm25_local.fit(all_embed_texts)
        sparse_vecs = bm25_local.encode_documents(new_texts)

        # 5. Embed + upsert to Pinecone ───────────────────────────────────
        upsert_to_pinecone(children, sparse_vecs)
        print(f'   ✅ Pinecone  {len(children)} vectors upserted')

        # 6. Parents → PostgreSQL ─────────────────────────────────────────
        upsert_parents_to_postgres(parents, figure_urls)
        print(f'   ✅ Postgres  {len(parents)} parents stored')

        # 7. Save checkpoint ──────────────────────────────────────────────
        done_keys.add(key)
        success.append(path)
        save_checkpoint(done_keys)
        with open(ALL_EMBED_TEXTS_PATH, 'w') as f:
            json.dump(all_embed_texts, f)

        print(f'   ⏱  {time.time()-t0:.1f}s total')

    except Exception as e:
        print(f'   ❌ FAILED: {e}')
        failed.append({'paper': paper, 'error': str(e)})


print(f"\n{'='*60}")
print(f'Done.  Success: {len(success)}  Failed: {len(failed)}')
if failed:
    for f in failed:
        print(f"  ❌ {f['paper']['name']}: {f['error']}")

Starting. Existing corpus: 0 texts.

────────────────────────────────────────────────────────────
📄 unknown/inc jan 2024.pdf  [RPM: 0/14]
   ✅ Gemini  RBB3013 January 2024 (16 sub-q)  [107.4s]
   ✅ Chunks  4 parents  16 children
   ✅ Images  5 uploaded to Supabase


Starting. Existing corpus: 0 texts.

────────────────────────────────────────────────────────────
📄 unknown/inc jan 2024.pdf  [RPM: 0/14]
   ✅ Gemini  RBB3013 January 2024 (16 sub-q)  [107.4s]
   ✅ Chunks  4 parents  16 children
   ✅ Images  5 uploaded to Supabase


I0601 15:45:10.490665 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.490814 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.491046 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.491057 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.491060 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.491062 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.491067 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.491069 30407535 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0601 15:45:10.491070 30407535 ev_poll_posix.cc:593] FD from fork parent

  0%|          | 0/16 [00:00<?, ?it/s]

Starting. Existing corpus: 0 texts.

────────────────────────────────────────────────────────────
📄 unknown/inc jan 2024.pdf  [RPM: 0/14]
   ✅ Gemini  RBB3013 January 2024 (16 sub-q)  [107.4s]
   ✅ Chunks  4 parents  16 children
   ✅ Images  5 uploaded to Supabase


...

   ✅ Pinecone  13 vectors upserted
   ✅ Postgres  4 parents stored
   ⏱  179.1s total

────────────────────────────────────────────────────────────
📄 unknown/inc sept2024.pdf  [RPM: 0/14]
   ⚠️  Gemini returned non-JSON, retrying once... (Gemini JSON parse failed: Expecting property name enclosed in double quotes: line 1 column 2 (char 1). Raw preview: {fsd} = 1\text{ mA}$, is to be used for 0-2.5 V, 0-10 V, 0-50 V, 0-250 V and 0-1000 V multirange voltmeter."     *   **Sub-question 1.b.i:** "Evaluate the set of series resistances, $R_1$, $R_2$, $R_3$ and $R_4$, required for this design." Marks: [10 marks]. Type: calculation. Has Figure: FIGURE Q1b.     *   **Sub-question 1.b.ii:** "If the meter is to be further scaled to cater for 5000 V range, )
   ✅ Gemini  RBB3013 SEPTEMBER 2024 SEMESTER (13 sub-q)  [233.7s]
   ✅ Chunks  4 parents  13 children
   ✅ Images  6 uploaded to Supabase


  0%|          | 0/89 [00:00<?, ?it/s]

   ✅ Pinecone  13 vectors upserted
   ✅ Postgres  4 parents stored
   ⏱  244.8s total

────────────────────────────────────────────────────────────
📄 unknown/inc sept2025.pdf  [RPM: 0/14]
   ✅ Gemini  RBB3013 September 2025 (16 sub-q)  [221.7s]
   ✅ Chunks  4 parents  16 children
   ✅ Images  5 uploaded to Supabase


  0%|          | 0/105 [00:00<?, ?it/s]

   ✅ Pinecone  16 vectors upserted
   ✅ Postgres  4 parents stored
   ⏱  234.6s total

Done.  Success: 7  Failed: 4
  ❌ inc jan2023.pdf: Gemini JSON parse failed: Expecting property name enclosed in double quotes: line 1 column 2 (char 1). Raw preview: {1}{36\pi \times 10^9} \varepsilon_r \frac{A}{d}$ | $F = YA \frac{\Delta t}{t}$ | $E_0 = \frac{Q}{C} = \frac{K\Delta t}{C} = G\Delta t$ |  Table II: | y | 13.1 | 26.2 | 39.3 | 52.4 | 65.5 | 78.6 | | :--- | :--- | :--- | :--- | :--- | :--- | :--- | | x | 5 | 10 | 15 | 20 | 25 | 30 |  Table III: | y | 14.7 | 29.4 | 44.1 | 58.8 | 73.5 | 88.2 | | :--- | :--- | :--- | :--- | :--- | :--- | :--- | | x | 
  ❌ inc may2022.pdf: Gemini JSON parse failed: Expecting property name enclosed in double quotes: line 1 column 2 (char 1). Raw preview: {d^2v(t)}{dt^2} + 5\frac{dv(t)}{dt} + 3v(t) = \frac{di(t)}{dt} + 5i(t)$. Determine the transfer function of the spring-mass damper system represented by the differential equation." Marks: 6. Type: calculation.

In [23]:
# ── Retry failed papers ───────────────────────────────────────────────────────
if not failed:
    print('No failures to retry.')
else:
    print(f'Retrying {len(failed)} papers...')
    retry_failed = []
    for entry in failed:
        paper = entry['paper']
        print(f"  ↩️  {paper['name']}")
        try:
            extracted         = extract_pdf(paper['path'])
            meta              = extracted['metadata']
            parents, children = build_chunks(extracted)
            new_texts         = [c.embed_text for c in children]
            figure_urls       = extract_and_upload_images(
                paper['path'], extracted['questions'],
                meta['course_code'], meta['semester'], meta['year']
            )
            all_embed_texts.extend(new_texts)
            bm25_r = BM25Encoder(); bm25_r.fit(all_embed_texts)
            upsert_to_pinecone(children, bm25_r.encode_documents(new_texts))
            upsert_parents_to_postgres(parents, figure_urls)
            done_keys.add(paper['key'])
            save_checkpoint(done_keys)
            print('     ✅ retry ok')
        except Exception as e:
            print(f'     ❌ still failing: {e}')
            retry_failed.append(entry)
    failed = retry_failed
    print(f'Remaining failures: {len(failed)}')

Retrying 3 papers...
  ↩️  inc may2022.pdf
   ⚠️  Gemini returned non-JSON, retrying... (JSON object does not start with a quoted key)
   ⚠️  Gemini still non-JSON, attempting repair... (JSON object does not start with a quoted key)


  0%|          | 0/130 [00:00<?, ?it/s]

     ✅ retry ok
  ↩️  inc may2023.pdf


  0%|          | 0/143 [00:00<?, ?it/s]

     ✅ retry ok
  ↩️  inc sept2022.pdf
   ⚠️  Gemini returned non-JSON, retrying... (JSON object does not start with a quoted key)
   ⚠️  Gemini still non-JSON, attempting repair... (JSON object does not start with a quoted key)


  0%|          | 0/158 [00:00<?, ?it/s]

     ✅ retry ok
Remaining failures: 0


---
## Section 6 — Refit BM25 on Full Corpus

Run once after all papers are ingested.  
The saved model is what `02_retrieval.ipynb` loads at query time.

In [24]:
print(f'Fitting BM25 on {len(all_embed_texts)} texts...')
bm25_final = BM25Encoder()
bm25_final.fit(all_embed_texts)
with open(BM25_PATH, 'wb') as f:
    pickle.dump(bm25_final, f)
print(f'✅ BM25 model saved → {BM25_PATH}')
print('   Reload in 02_retrieval.ipynb (re-run the BM25 load cell).')

Fitting BM25 on 158 texts...


  0%|          | 0/158 [00:00<?, ?it/s]

✅ BM25 model saved → data/bm25_model.pkl
   Reload in 02_retrieval.ipynb (re-run the BM25 load cell).


---
## Section 7 — Report

In [25]:
conn = psycopg2.connect(os.getenv('DATABASE_URL'))
cur  = conn.cursor()
cur.execute("""
    SELECT course_code, semester, year,
           COUNT(*) as questions,
           COUNT(CASE WHEN image_urls != '{}' THEN 1 END) as with_images
    FROM   parent_chunks
    GROUP BY course_code, semester, year
    ORDER BY course_code, year DESC;
""")
rows = cur.fetchall(); cur.close(); conn.close()
stats = index.describe_index_stats()

print('=' * 62)
print('PAPERSLOTH — INGESTION REPORT')
print('=' * 62)
print(f'{"SUBJECT":<12} {"SEMESTER":<22} {"YEAR":<6} {"Qs":<6} {"w/img"}')
print('-' * 62)
for r in rows:
    print(f'{r[0]:<12} {r[1]:<22} {r[2]:<6} {r[3]:<6} {r[4]}')
print('-' * 62)
print(f'Total questions in PostgreSQL : {sum(r[3] for r in rows)}')
print(f'Total vectors in Pinecone     : {stats["total_vector_count"]}')
print(f'BM25 corpus size              : {len(all_embed_texts)} texts')

PAPERSLOTH — INGESTION REPORT
SUBJECT      SEMESTER               YEAR   Qs     w/img
--------------------------------------------------------------
EDB2613      MAY 2024 SEMESTER      2024   4      4
EDB2613      MAY 2023 SEMESTER      2023   4      4
EDB2613      JANUARY 2023 SEMESTER  2023   4      2
EDB2613      SEPTEMBER 2023 SEMESTER 2023   4      4
EDB2613      September 2022         2022   4      2
EDB2613      MAY 2022 SEMESTER      2022   5      4
RBB3013      May 2025               2025   4      4
RBB3013      JANUARY 2025 SEMESTER  2025   4      3
RBB3013      September 2025         2025   4      4
RBB3013      SEPTEMBER 2024 SEMESTER 2024   4      4
RBB3013      January 2024           2024   4      3
--------------------------------------------------------------
Total questions in PostgreSQL : 45
Total vectors in Pinecone     : 158
BM25 corpus size              : 158 texts
